# Agent hạ cánh LunarLander (portfolio)

Bạn tự thiết kế, train và đánh giá một agent cho `LunarLander-v3`, rồi quay video nó hạ cánh. Notebook chỉ chấm 4 hàm tiện ích
ở mục 1 (đánh giá, video, đường cong, IQM); phần agent là thiết kế của bạn. Gợi ý mặc định: PPO của bài Intermediate 02 với
cấu hình cột LunarLander ở mục 6 phần lý thuyết.

- Chấm: `uv run pytest tracks/ai-engineer/modules/07-reinforcement-learning/exercises/advanced/01-lunarlander-agent -q`
- Video và trọng số **không commit**: ghi vào `starter/videos/` và `starter/checkpoints/` (hoặc nơi khác ngoài git).

In [ ]:
# explore: giới hạn luồng, tìm gốc repo
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
HERE = Path.cwd()
print(ROOT, HERE)

In [ ]:
import time
from pathlib import Path

import gymnasium as gym
import imageio
import numpy as np
import torch
import torch.nn as nn

## 1. Bốn hàm được chấm

- `evaluate_policy(policy, make_env, n_episodes, seed)`: như Intermediate 01. Một env, episode `i` reset với `seed + i`,
  `policy(obs) -> action`; trả mảng numpy tổng reward.
- `record_video(policy, make_env, path, seed, fps=30)`: `make_env()` trả env có `render_mode="rgb_array"`. Chạy **một** episode
  (reset với `seed`), lấy một khung `env.render()` sau reset và một khung sau mỗi `step`, ghi ra `path` bằng
  `imageio.mimsave(path, frames, fps=fps, macro_block_size=1)` (tạo thư mục cha nếu chưa có; đuôi `.mp4` hoặc `.gif`). Trả
  `{"frames": số khung, "steps": số bước, "return": tổng reward}`.
- `curve_on_grid(end_steps, returns, grid, window)`: với mỗi mốc `g` trong `grid`, trung bình return của các episode kết thúc
  trong `(g - window, g]` bước env; không có episode nào thì `nan`. Dùng để đưa đường cong của nhiều seed về cùng trục.
- `interquartile_mean(x)`: IQM (Agarwal 2021): sắp xếp, bỏ `floor(0.25 · n)` phần tử ở mỗi đầu, trung bình phần còn lại.

In [ ]:
def evaluate_policy(policy, make_env, n_episodes: int, seed: int) -> np.ndarray:
    # TODO: một env từ make_env(); episode i reset với seed + i; trả mảng tổng reward
    raise NotImplementedError


def record_video(policy, make_env, path, seed: int, fps: int = 30) -> dict:
    # TODO: make_env() trả env có render_mode="rgb_array"; một khung sau reset và một khung sau mỗi step;
    # ghi bằng imageio (tạo thư mục cha nếu chưa có); trả {"frames", "steps", "return"}
    raise NotImplementedError


def curve_on_grid(end_steps, returns, grid, window: int) -> np.ndarray:
    # TODO: với mỗi g trong grid: trung bình return các episode có end_step trong (g - window, g]; không có thì nan
    raise NotImplementedError


def interquartile_mean(x) -> float:
    # TODO: sắp xếp, bỏ floor(25%) phần tử mỗi đầu, lấy trung bình phần còn lại
    raise NotImplementedError

## 2. Agent của bạn

Thêm các cell (định nghĩa hàm, lớp) cho agent: tự viết, không dùng thư viện RL. Yêu cầu tối thiểu:

- Train ít nhất **3 seed** với cùng cấu hình, mỗi seed ghi lịch sử `(end_steps, returns)` và (nếu là PPO) approx KL, clip fraction,
  entropy theo từng lần cập nhật.
- Đánh giá định kỳ trên một tập seed riêng (ví dụ 10 episode, seed 20 000..20 009) để chọn checkpoint; đánh giá cuối trên
  **100 episode cố định** (seed 10 000..10 099), policy greedy.
- Lưu checkpoint (`torch.save(model.state_dict(), ...)`) ra ngoài git.

Cấu hình laptop (CPU, không cần GPU) và thời gian chạy đo được ghi trong README của bài.

In [ ]:
# explore: mốc so sánh: policy ngẫu nhiên và policy "không làm gì" trên 100 episode đánh giá
make_env = lambda: gym.make("LunarLander-v3")
rng = np.random.default_rng(0)
for name, pol in [("ngẫu nhiên", lambda o: int(rng.integers(4))), ("không làm gì", lambda o: 0)]:
    r = evaluate_policy(pol, make_env, 100, 10_000)
    print(f"{name:13s}: trung bình {r.mean():8.1f} | IQM {interquartile_mean(r):8.1f} | tỉ lệ >= 200: {np.mean(r >= 200):.2f}")

In [ ]:
# explore: TODO: train agent của bạn trên 3 seed, vẽ đường cong (curve_on_grid + trung bình và min-max giữa seed),
# đánh giá 100 episode cố định, quay video bằng record_video với gym.make("LunarLander-v3", render_mode="rgb_array")